In [1]:
#importing libraries
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import json
import os
import csv

In [3]:
#!pip install tf-keras

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 15.6 MB/s eta 0:00:00


In [3]:
import shutil
from sentence_transformers import SentenceTransformer

W0728 14:27:38.843000 8740 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
from llama_cpp import Llama

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import re
from datetime import datetime
from paddleocr import PaddleOCR
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from textwrap import wrap
from reportlab.lib.utils import ImageReader

In [6]:
# === paths of all files ===
base_dir = r"C:\Users\hp\Downloads\Project folder of Chatter"
faiss_path = os.path.join(base_dir, "faiss_index_PaddleOCR.index")
csv_path = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2/chunked_text_PaddleOCR.csv")
text_folder = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2")
logo_path = os.path.join(base_dir, "Logo.png")
mistral_path = r"C:\Users\hp\Downloads\mistral-7b-instruct-v0.1.Q4_K_M.gguf"
CACHE_FILE = "cache.json"
FEEDBACK_FILE = "feedback_log.csv"

In [7]:
# === Load all components ===

# Load the FAISS index from disk — used for efficient similarity search over embedded text chunks
index = faiss.read_index(faiss_path)

# Load the preprocessed and chunked text data (from OCR'd documents) into a DataFrame
df_chunks = pd.read_csv(csv_path)

# Load the sentence embedding model — used to convert user queries and text chunks into vector form
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load the local LLM (e.g., Mistral 7B in GGUF format) with context window size set to 2048 tokens
llm = Llama(model_path=mistral_path, n_ctx=2048)

# Initialize the PaddleOCR model with English language and angle classification enabled
ocr_model = PaddleOCR(use_angle_cls=True, lang='en')

llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from C:\Users\hp\Downloads\mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:     

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None)
The model(UVDoc) is not supported to run in MKLDNN mode! Using `paddle` instead!
Using official model (UVDoc), the model files will be automatically downloaded and saved in C:\Users\hp\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in C:\Users\hp\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv5_server_det', None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in C:\Users\hp\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv5_server_rec', None)
Using official model (PP-OCRv5_server_rec), the model files will be automatically downloaded and saved in C:\Users\hp\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [8]:
# === Session Logs ===
cache = {}
chat_log = []

In [9]:
# === Cache ===

def load_cache():
    # Load cache from file if it exists
    return json.load(open(CACHE_FILE, "r", encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

def save_cache(cache):
    # Save cache to file as JSON
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4, ensure_ascii=False)


In [10]:
def save_feedback(query, answer, source, feedback):
    # Prepare feedback data as a dictionary
    data = {"query": query, "answer": answer, "source": source, "feedback": feedback}
    
    # Check if feedback file already exists
    file_exists = os.path.isfile(FEEDBACK_FILE)
    
    # Open feedback file in append mode
    with open(FEEDBACK_FILE, "a", encoding="utf-8", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data.keys())
        
        # Write header if file is new
        if not file_exists:
            writer.writeheader()
        
        # Write feedback data to file
        writer.writerow(data)


In [11]:
# === Reset everything ===

def reset_all():
    """Clear cache, feedback log, temporary docs, and chat history."""
    # Clear cache file by writing empty JSON object
    open(CACHE_FILE, "w").write("{}")
    # Clear feedback log file with header row
    open(FEEDBACK_FILE, "w").write("query,answer,source,feedback\n")
    # Delete temporary documents folder, ignore errors if not present
    shutil.rmtree("temp_docs", ignore_errors=True)
    # Clear in-memory chat log
    chat_log.clear()
    # Return placeholders for UI components or variables to reset
    return [], None, None, None, None, None


In [12]:
# === OCR extraction ===

def extract_text_from_image(image_path):
    # Run OCR on image and get results
    ocr_result = ocr_model.ocr(image_path, cls=True)
    # Extract text lines from OCR output and join with newlines
    extracted_text = "\n".join([line[1][0] for block in ocr_result for line in block])
    return extracted_text


In [13]:
# === FAISS retrieval (+ optional OCR chunk) ===

# Perform semantic search over pre-embedded text chunks using a FAISS index
# Optionally, include extra text (e.g., OCR from an uploaded image) as an additional result
def search_top_k(query, k=2, extra_text=None):
    # Convert the query into an embedding vector using the sentence transformer
    embedding = embedding_model.encode([query])
    
    # Search the FAISS index to find the top-k most similar vectors (chunks)
    distances, indices = index.search(np.array(embedding).astype("float32"), k)
    
    # Retrieve the corresponding text chunks (with metadata) from the DataFrame
    faiss_results = df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]]
    
    # If additional text (like from OCR) is provided, add it as an extra row in the results
    if extra_text:
        extra_df = pd.DataFrame([{
            "chunk_id": "ocr_chunk",                # Identifier for the dynamic chunk
            "filename": "uploaded_image",           # Label the source as image input
            "chunk_text": extra_text                # Actual OCR-extracted text
        }])
        faiss_results = pd.concat([faiss_results, extra_df], ignore_index=True)

    # Return a DataFrame containing top-k retrieved chunks + optional extra text
    return faiss_results


In [14]:
# === Re-rank retrieved chunks using cosine similarity to refine semantic relevance ===

# Given a set of retrieved chunks, re-rank them by computing cosine similarity between
# the user's question and each chunk to improve semantic accuracy.
def rerank_chunks(chunks_df, question, model):
    # Generate the embedding vector for the input question
    q_vec = model.encode([question])[0]

    # Compute cosine similarity between the question vector and each chunk's embedding
    # Add a new column 'score' to store similarity values
    chunks_df["score"] = chunks_df["chunk_text"].apply(
        lambda text: cosine_similarity([q_vec], [model.encode([text])[0]])[0][0]
    )

    # Sort the chunks in descending order of similarity score and return the top 2
    return chunks_df.sort_values("score", ascending=False).head(2)


In [15]:
# === Best semantic snippet ===

# From a set of retrieved document chunks, identify the single sentence that is most semantically
# similar to the user's question, using cosine similarity on sentence embeddings.
def find_best_semantic_snippet(chunks_df, question, model, max_length=250):
    # Generate embedding for the user's question
    question_vec = model.encode([question])[0]

    # Initialize variables to track the best-matching sentence
    best_snippet, best_file, best_score = "", "", -1

    # Iterate through each chunk (row) in the DataFrame
    for _, row in chunks_df.iterrows():
        # Split chunk text into individual sentences
        sentences = re.split(r'(?<=[.!?]) +', row["chunk_text"])
        
        for sent in sentences:
            sent = sent.strip()
            
            # Skip very short sentences (likely noise)
            if len(sent) < 10:
                continue

            # Compute cosine similarity between question and this sentence
            score = cosine_similarity([question_vec], [model.encode([sent])[0]])[0][0]

            # Update best match if current sentence is more similar
            if score > best_score:
                best_score = score
                best_snippet = sent
                best_file = row["filename"]

    # If no suitable sentence was found, return a fallback (first chunk, truncated)
    if not best_snippet:
        return chunks_df.iloc[0]["filename"], chunks_df.iloc[0]["chunk_text"][:max_length].strip() + "..."

    # Return the best-matching filename and sentence, truncated if too long
    return best_file, best_snippet if len(best_snippet) < max_length else best_snippet[:max_length].strip() + "..."


In [57]:
def build_prompt(chunks, question, chat_log=None):
    # Determine context type
    if isinstance(chunks, list):
        context = "\n".join(chunks)
    elif hasattr(chunks, "chunk_text"):
        context = "\n".join(chunks["chunk_text"].tolist())
    else:
        context = ""

    # Build history text (unchanged)
    history_text = ""
    if chat_log:
        for i, entry in enumerate(chat_log[-2:], start=1):
            history_text += f"Q{i}: {entry['question']}\nA{i}: {entry['answer']}\n"

    prompt = (
        "You are a helpful assistant. Answer the user's question using the provided context. "
        "If unsure, say 'I don't know'.\n\n"
        "### Prior Conversation ###\n" + history_text +
        "### Context from Documents ###\n" + context + "\n\n" +
        "### Question ###\n" + question + "\n\n"
        "### Answer ###\n"
    )
    return prompt


In [ ]:
import os

# Set your Hugging Face API key in the environment variable
os.environ["HUGGINGFACE_API_KEY"] = "myAPIkey"

# Retrieve the API key from environment variable
api_key = os.getenv("HUGGINGFACE_API_KEY")
print("Key:", api_key)

# Prepare headers for the API call
headers = {
    "Authorization": f"Bearer {api_key}"
}


In [ ]:
import os
import requests

# Hugging Face remote API call
def query_huggingface(prompt, model="gpt2"):
    api_key = os.getenv("HUGGINGFACE_API_KEY")
    if not api_key:
        raise ValueError("Hugging Face API key not found.")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "inputs": prompt,
        "parameters": {"max_new_tokens": 200}
    }

    response = requests.post(
        f"https://api-inference.huggingface.co/models/gpt2",
        headers=headers,
        json=payload
    )

    if response.status_code == 200:
        return response.json()[0]["generated_text"]
    else:
        raise RuntimeError(f"Hugging Face API failed: {response.status_code} - {response.text}")

# Full generation logic
def generate_answer(chunks, question, chat_log=None):
    prompt = build_prompt(chunks, question, chat_log)

    try:
        #  Primary: Try Hugging Face
        answer = query_huggingface(prompt)
        return answer.strip()

    except Exception as e:
        print(" Hugging Face API failed. Falling back to local LLM...", e)
        try:
            #  Fallback: Local model like Mistral
            response = llm(prompt, max_tokens=200, stop=["</s>", "###"])
            return response["choices"][0]["text"].strip()

        except Exception as inner_e:
            print(" Local model also failed:", inner_e)
            return "Sorry, both Hugging Face and local models failed."


In [61]:
query = "Who is Marieve Herington?"

# Step 1: Retrieve relevant chunks from FAISS index
relevant_chunks = search_top_k(query, k=3)

# Step 2: Generate answer using those chunks
answer = generate_answer(relevant_chunks, query, chat_log)

print("Answer:", answer)

 Hugging Face API failed. Falling back to local LLM... Hugging Face API failed: 404 - Not Found


Llama.generate: 529 prefix-match hit, remaining 460 prompt tokens to eval
llama_perf_context_print:        load time =  270149.80 ms
llama_perf_context_print: prompt eval time =  595932.07 ms /  1433 tokens (  415.86 ms per token,     2.40 tokens per second)
llama_perf_context_print:        eval time =  100232.27 ms /   165 runs   (  607.47 ms per token,     1.65 tokens per second)
llama_perf_context_print:       total time =  184692.03 ms /  1598 tokens


Answer: Marieve Herington is a Canadian Marieve Herington actress who has appeared in recurring roles on How I Met Your Your Mother , Good Luck Charlie and Ever After High . She provides the voice of Tilly Green on the Disney Channel show and has also voiced lead characters in several animated shows and video games, including Celestia Ludenburg in Danganronpa : Trigger Happy Havoc, Otter in Franklin and Fifi La Fume in Tiny Toons Looniversity. She began singing in major public performances at the age of 12 and has been fronting her own jazz ensembles since the age of 16. She studied drama at the University of Toronto and is a member of the Alliance of Canadian Cinema Television and Radio Artists (ACTRA).


In [65]:
# Handles user feedback submission by saving it and updating the UI history
def feedback_fn(query, answer, source, feedback, history):
    # Save the feedback to the feedback log (CSV file)
   save_feedback(query, answer, source, feedback)
    # Append a feedback confirmation message to the chat history
   return history + [(" Feedback received: " + feedback.upper(), "")]

# Retrieves the count of good/bad feedback for a given query from the feedback log
def get_feedback_score(query):
    # If the feedback file does not exist, return a default message
    if not os.path.exists(FEEDBACK_FILE):
        return "No feedback yet."
    # Load the feedback log into a DataFrame
    df = pd.read_csv(FEEDBACK_FILE)
    # Filter for rows that match the given query
    match = df[df["query"] == query]
    # If no matching feedback is found, return a default message
    if match.empty:
        return "No feedback yet."
    # Count the number of 'good' and 'bad' feedback entries
    good = match[match["feedback"] == "good"].shape[0]
    bad = match[match["feedback"] == "bad"].shape[0]
     # Return a summary of the feedback counts
    return f"Feedback: 👍 {good} | 👎 {bad}"


In [66]:
# Exports the current chat log to a CSV file with a timestamped filename
def export_chat_to_csv():
    from gradio import update  # Import update for dynamic UI changes

    # If chat_log is empty, hide the export download component
    if not chat_log:
        return update(value=None, visible=False)

    # Generate a timestamped filename for the export
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"chat_export_{ts}.csv"

    # Convert the chat_log (a list of tuples or dicts) into a DataFrame and save as CSV
    pd.DataFrame(chat_log).to_csv(path, index=False)

    # Make the export download component visible and set its file path
    return update(value=path, visible=True)

# Exports the feedback log CSV (if it exists) by copying it to a new timestamped file
def export_feedback_to_csv():
    from gradio import update  # Import update for dynamic UI changes

    # If feedback file does not exist, hide the export download component
    if not os.path.exists(FEEDBACK_FILE):
        return update(value=None, visible=False)

    # Generate a timestamped filename for the feedback export
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    new_path = f"feedback_export_{ts}.csv"

    # Copy the existing feedback file to the new path
    shutil.copy(FEEDBACK_FILE, new_path)

    # Make the export download component visible and set its file path
    return update(value=new_path, visible=True)


In [72]:
# === Main chat function ===
def chatbot_ui(query, history, image_file, chat_log=None):
    # Initialize chat_log if None
    if chat_log is None:
        chat_log = []
    cache = load_cache()  # Load cached responses if any

    # Extract text from image if uploaded
    ocr_text = extract_text_from_image(image_file.name) if image_file else None

    # Check if the query is already in cache
    if query in cache:
        answer = cache[query]["answer"]
        source = cache[query]["source"]
        snippet = cache[query]["snippet"]
        filepath = os.path.join("temp_docs", source)  # Path to cached source file
    else:
        # Search top-k chunks from documents (includes OCR text if present)
        chunks_raw = search_top_k(query, k=5, extra_text=ocr_text)

        # Re-rank the chunks based on semantic similarity
        chunks = rerank_chunks(chunks_raw, query, embedding_model)

        # Generate an answer using the top-ranked chunks
        answer = generate_answer(chunks, query, chat_log)

        # Identify the best matching document snippet and source
        source, snippet = find_best_semantic_snippet(chunks, query, embedding_model)

        # Copy the source file to a temporary location for download
        local_file_path = os.path.join(text_folder, source)
        os.makedirs("temp_docs", exist_ok=True)  # Ensure temp_docs directory exists
        filepath = os.path.join("temp_docs", source)
        if os.path.exists(local_file_path):
            shutil.copy(local_file_path, filepath)
        else:
            filepath = None  # If file doesn't exist, disable file output

        # Cache the new query response
        cache[query] = {"answer": answer, "source": source, "snippet": snippet}
        save_cache(cache)

    # Add Q&A with source and snippet to chat log
    chat_log.append({"question": query, "answer": answer, "source": source, "snippet": snippet})
    feedback_summary = get_feedback_score(query)
    display = f" Answer: {answer}\n Source: {source}\n Snippet: {snippet}\n {feedback_summary}"

    # Add this round of interaction to chat history
    history.append((query, display))

    from gradio import update
    # Show download button for the source file if it exists, otherwise hide
    file_output = filepath if filepath and os.path.exists(filepath) else update(visible=False)

    # Return updated history and outputs
    return history, query, answer, source, file_output


In [73]:
# === NEW: Export chat to PDF ===

def export_chat_to_pdf():
    """Generate a nicely formatted PDF of the entire chat_log."""
    from gradio import update
    if not chat_log:
        return update(value=None, visible=False)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_path = f"chat_export_{ts}.pdf"
    c = canvas.Canvas(pdf_path, pagesize=A4)
    width, height = A4

    # Header
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, height - 60, "Chatter – Chat Summary Report")
    c.setFont("Helvetica", 10)
    c.drawString(50, height - 75, f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # Logo (optional)
    if os.path.exists(logo_path):
        try:
            c.drawImage(ImageReader(logo_path), width - 100, height - 100, width=40, preserveAspectRatio=True, mask='auto')
        except Exception:
            pass

    # Body
    y = height - 130
    c.setFont("Helvetica", 11)
    for i, entry in enumerate(chat_log, start=1):
        block = [
            f"{i}. Question: {entry['question']}",
            f"   Answer: {entry['answer']}",
            f"   Source: {entry['source']}",
            f"   Snippet: {entry['snippet']}",
        ]
        for line in block:
            for wrap_line in wrap(line, width=100):
                if y < 80:
                    c.showPage()
                    c.setFont("Helvetica", 11)
                    y = height - 80
                c.drawString(50, y, wrap_line)
                y -= 16
            y -= 6
    c.save()
    return update(value=pdf_path, visible=True)


In [74]:
# === Feedback summary loader ===

def load_feedback_summary():
    if not os.path.exists(FEEDBACK_FILE):
        return pd.DataFrame(columns=["query", "answer", "source", "feedback"])
    return pd.read_csv(FEEDBACK_FILE)

In [75]:
# === Gradio UI ===
with gr.Blocks(title="Chatter 1.5 – PDF Enhanced") as demo:
    gr.Markdown("## 🤖 Chatter 1.5 – Complete Chatbot with Feedback & PDF Export")

    chatbot = gr.Chatbot()
    query = gr.Textbox(label="Ask something...")
    image_input = gr.File(label="Upload Image for OCR", file_types=["image"])
    file_viewer = gr.File(label="📄 View Full Document", visible=True)

    # Export / download widgets
    export_csv_btn = gr.Button("💾 Export Chat to CSV")
    export_file = gr.File(label="📥 Download Chat CSV", visible=False)

    export_pdf_btn = gr.Button("🧾 Export Chat to PDF")
    pdf_file = gr.File(label="📄 Download Chat PDF", visible=False)

    export_feedback_btn = gr.Button("📊 Export Feedback Log")
    export_feedback_file = gr.File(label="📥 Download Feedback CSV", visible=False)

    feedback_summary_btn = gr.Button("📈 Show Feedback Summary")
    feedback_table = gr.DataFrame(headers=["query", "answer", "source", "feedback"], interactive=False, visible=False)

    with gr.Row():
        btn_submit = gr.Button("Submit")
        btn_reset = gr.Button("♻️ Reset Chat & Cache")

    with gr.Row():
        btn_good = gr.Button("👍")
        btn_bad = gr.Button("👎")

    # State vars
    state_query = gr.State()
    state_answer = gr.State()
    state_source = gr.State()
    state_file = gr.State()

    # --- Callbacks ---
    btn_submit.click(chatbot_ui, inputs=[query, chatbot, image_input], outputs=[chatbot, state_query, state_answer, state_source, file_viewer])

    btn_reset.click(reset_all, inputs=[], outputs=[chatbot, file_viewer, export_file, export_feedback_file, feedback_table, pdf_file])

    btn_good.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="good"), chatbot], outputs=[chatbot])
    btn_bad.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="bad"), chatbot], outputs=[chatbot])

    export_csv_btn.click(export_chat_to_csv, inputs=[], outputs=[export_file])
    export_feedback_btn.click(export_feedback_to_csv, inputs=[], outputs=[export_feedback_file])
    export_pdf_btn.click(export_chat_to_pdf, inputs=[], outputs=[pdf_file])

    feedback_summary_btn.click(load_feedback_summary, inputs=[], outputs=[feedback_table])
    feedback_summary_btn.click(lambda: gr.update(visible=True), None, [feedback_table])


C:\Users\hp\AppData\Local\Temp\ipykernel_8740\3221585447.py:5: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


In [76]:
# === Launch ===
if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


 Hugging Face API failed. Falling back to local LLM... Hugging Face API failed: 404 - Not Found


Llama.generate: 46 prefix-match hit, remaining 1471 prompt tokens to eval
llama_perf_context_print:        load time =  270149.80 ms
llama_perf_context_print: prompt eval time =  318778.54 ms /  1471 tokens (  216.71 ms per token,     4.61 tokens per second)
llama_perf_context_print:        eval time =   10042.79 ms /    13 runs   (  772.52 ms per token,     1.29 tokens per second)
llama_perf_context_print:       total time =  329152.60 ms /  1484 tokens


 Hugging Face API failed. Falling back to local LLM... Hugging Face API failed: 404 - Not Found


Llama.generate: 46 prefix-match hit, remaining 1326 prompt tokens to eval
llama_perf_context_print:        load time =  270149.80 ms
llama_perf_context_print: prompt eval time =  257625.44 ms /  1326 tokens (  194.29 ms per token,     5.15 tokens per second)
llama_perf_context_print:        eval time =    4192.58 ms /     6 runs   (  698.76 ms per token,     1.43 tokens per second)
llama_perf_context_print:       total time =  261877.13 ms /  1332 tokens
